# 정정과 후속 질문을 받는 공지 챗봇

지난 실습에서 한 번의 요청에 목적과 원문을 넣었습니다. 이번에는 사용자가 “날짜만 바꿔 줘”라고 했을 때 장소와 시간이 사라지지 않게 대화를 이어 갑니다. 먼저 노트북에서 대화를 보내고, 같은 호출 함수를 채팅 화면에 연결합니다.

다운로드한 노트북은 새 수업 폴더에서 여세요. 아래 셀은 그 폴더에 수업용 파일 두 개를 씁니다. Python 3 커널과 앞 단원의 OpenAI API 키를 사용합니다. 수업용 로컬 환경에서 그 키를 환경 변수 OPENAI_API_KEY에 설정하거나, 노트북과 같은 폴더의 .env에서 읽게 하세요. 키는 노트북에 적지 않습니다. 모델 호출 셀을 다시 실행하면 새 요청이 발생합니다. 새 커널이라면 아래 키 입력 셀을 실행하세요. 입력한 값은 이 커널과 여기서 띄운 채팅 앱에서만 사용합니다.

## 11일 · 날짜만 정정하고 장소는 유지하기

동아리 공지 C4를 보세요. 첫 말에는 시간과 장소가 모두 있습니다. 두 번째 말에는 날짜만 새로 나옵니다. 세 번째 말은 “최종 모임 일시와 장소를 한 문장으로 정리해 주세요”입니다. 세 번째 문장만 받은 사람은 언제 어디에서 모이는지 답할 수 있을까요?

모델에게도 앞에서 한 말을 함께 보냅니다. 역할이 user인 메시지는 사용자가 한 말이고, assistant는 모델이 한 답입니다. 두 역할의 말을 오간 순서대로 목록에 보관합니다. 요청할 때마다 그 목록과 새 질문을 함께 보냅니다. 여기서 ‘새 대화’는 빈 목록으로 시작하는 대화입니다.

### 수업 폴더에 호출 함수 준비하기

수업 커널에 필요한 패키지를 설치합니다. 이미 설치되어 있으면 다음 셀로 넘어가세요. 설치 직후 가져오기에 실패하면 커널을 재시작하세요.

In [ ]:
%pip install "openai>=2,<4" "streamlit>=1.40,<2" python-dotenv

In [ ]:
import os
from getpass import getpass
from pathlib import Path
from dotenv import load_dotenv

load_dotenv(Path.cwd() / ".env")  # 수업 폴더에 키 파일이 있다면 읽습니다.
if not os.getenv("OPENAI_API_KEY"):
    os.environ["OPENAI_API_KEY"] = getpass("수업용 OpenAI API 키: ")  # 입력한 키는 화면에 표시되지 않습니다.

노트북과 화면 양쪽에서 같은 함수를 쓰도록 다음 셀을 한 번 실행하세요. 첫 줄의 %%writefile은 셀 내용을 공지 도우미 파일에 저장하는 Jupyter 명령입니다. 같은 이름의 파일이 있으면 덮어씁니다. 먼저 instructions를 읽어 보세요. “정정한 항목”과 “그대로 둔 항목”을 구분하면 날짜만 바꾸는 요청의 범위가 명확해집니다.

In [ ]:
%%writefile notice_chat.py
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv()
client = OpenAI()  # 환경에 설정한 키를 읽습니다.
RULE = "소리채집 동아리 공지 도우미입니다. 사용자가 정정한 항목은 최신 내용을 사용하고 그대로 둔 항목은 유지하세요. 모임 정보가 없으면 어떤 정보가 필요한지 질문하세요."

def send(history, question):
    user = {"role": "user", "content": question}
    response = client.responses.create(
        model="gpt-4.1-mini",
        instructions=RULE,
        input=history + [user],  # 앞선 대화 뒤에 새 질문을 붙여 보냅니다.
        store=False,
    )
    reply = response.output_text
    # 응답을 받은 뒤에만 한 쌍을 기록하므로 실패한 질문은 이력에 남지 않습니다.
    history.extend([user, {"role": "assistant", "content": reply}])
    return reply

### 첫 공지 보내기

새 목록을 만들고 C4의 첫 발화를 보냅니다. 출력에서 **5월 21일·오후 4시·가온관 305호**를 찾아 보세요. 아래 출력은 2026년 11월 26일 gpt-4.1-mini를 실제 호출한 결과입니다. 답변의 문장과 줄바꿈은 실행할 때마다 달라질 수 있습니다.

In [3]:
from notice_chat import send
history = []  # 이 셀부터 다시 실행하면 빈 대화로 시작합니다.
print(send(history, "모임은 5월 21일 금요일 오후 4시예요. 가온관 305호에서 만나요."))

소리채집 동아리 모임 안내입니다.

일시: 5월 21일 금요일 오후 4시  
장소: 가온관 305호

더 필요한 정보가 있으면 알려주세요!


첫 공지에 대한 답입니다. 이제 장소를 다시 입력하지 않고 날짜만 고칩니다. 첫 셀을 다시 실행하지 말고 바로 아래 셀로 가세요.

In [4]:
print(send(history, "날짜를 정정할게요. 5월 22일 토요일입니다. 시간과 장소는 그대로예요."))

수정된 소리채집 동아리 모임 안내입니다.

일시: 5월 22일 토요일 오후 4시  
장소: 가온관 305호

다른 문의 사항 있으시면 알려주세요!


날짜는 22일로 바뀌고 오후 4시와 305호는 남았습니다. 이제 원문 사실을 다시 쓰지 않고 답변 형식만 지정해 봅니다.

In [5]:
last_question = "최종 모임 일시와 장소를 한 문장으로 정리해 주세요."
print(send(history, last_question))

소리채집 동아리 모임은 5월 22일 토요일 오후 4시에 가온관 305호에서 진행됩니다.


### 실습 · 똑같은 마지막 질문을 새 대화에 보내기

기존 history는 그대로 보관하세요. 다른 빈 목록을 만들고 방금 쓴 last_question만 보내세요. 답에 305호가 있다면 그 방 번호를 알려 준 메시지가 실제 요청에 들어갔는지 확인하세요. 출력 문장을 바탕으로 동료에게 “초기화하면 어떤 정보가 사라지는가?”를 설명하세요.

In [ ]:
# 여기에 새 목록으로 마지막 질문을 보내고 답을 출력하세요.

<details><summary>모범 예시와 확인 기준</summary><pre><code>fresh = []
print(send(fresh, last_question))

# 수업에서 실제로 받은 응답
# 최종 모임 일시와 장소 정보를 알려주시면 한 문장으로 정리해 드리겠습니다.</code></pre><p>새 목록에는 지난 공지가 한 건도 없습니다. 모임 정보를 되묻는 답이 근거에 맞습니다. 기존 history를 또 넘겼다면 새 대화가 시작되지 않습니다.</p></details>

## 12일 · 질문이 들어올 때만 응답하는 채팅 화면

노트북에서는 원하는 셀만 골라 실행했습니다. Streamlit은 사용자가 입력을 보내거나 체크박스를 바꾸면 앱 파일을 위에서부터 다시 실행합니다. 파일 맨 위에 history = []를 쓰면 이전 대화가 사라집니다. st.session_state는 같은 브라우저 탭에서 다시 실행할 때에도 값을 보관하므로 대화 목록을 여기에 둡니다.

화면에서 할 일은 세 가지입니다. 먼저 기존 메시지를 그립니다. 새 입력을 확인합니다. 입력이 들어온 실행에서만 send를 부릅니다. <mark>지난 답을 다시 화면에 그리는 동안에는 모델에 다시 묻지 않습니다.</mark>

### 화면 파일 만들기

다음 셀에서 위쪽은 기존 메시지 표시, 아래쪽 if question은 새 입력 처리입니다. 채팅 입력을 보내지 않은 실행에서는 question에 질문이 없으므로 그 안을 건너뜁니다. 체크박스는 모델 호출 횟수와 대화 쌍 수를 확인하기 위한 것입니다.

In [ ]:
%%writefile notice_app.py
import streamlit as st
from notice_chat import send

st.title("소리채집 동아리 공지 도우미")
if "messages" not in st.session_state:
    st.session_state.messages = []
    st.session_state.calls = 0

# 13일 과제: 이곳에 새 대화 버튼과 초기화 동작을 추가하세요.
show_counts = st.checkbox("실행 정보 보기")
history = st.session_state.messages
for message in history:
    st.chat_message(message["role"]).write(message["content"])

question = st.chat_input("공지나 후속 질문을 입력하세요")
if question:
    st.session_state.calls += 1  # 보내기를 누른 입력만 집계합니다.
    st.chat_message("user").write(question)
    reply = send(history, question)
    st.chat_message("assistant").write(reply)

if show_counts:
    st.caption(f"모델 요청 {st.session_state.calls}건 · 저장된 대화 {len(history) // 2}쌍")

### 내 컴퓨터에서 채팅창 열기

아래 셀을 한 번 실행하고 브라우저에서 http://localhost:8501 을 여세요. 노트북의 Python과 같은 환경으로 서버를 시작합니다. 주소가 열리지 않으면 셀 아래에 나온 서버 오류를 확인하세요. 8501을 다른 수업 앱이 쓰고 있다면 여기의 포트 번호와 브라우저 주소를 함께 바꾸세요.

In [ ]:
import subprocess
import sys

server = subprocess.Popen([sys.executable, "-m", "streamlit", "run", "notice_app.py",
                           "--server.port", "8501", "--server.headless", "true"])

### 실습 · 두 번 보내고 체크박스만 바꾸기

‘실행 정보 보기’를 켜세요. 첫 입력으로 아래 C1 공지를 보내고, 두 번째 입력으로 “진행자는 몇 시까지 가야 하나요?”라고 물으세요. 그다음 체크박스를 껐다가 켜세요. 활동지에는 첫 입력, 두 번째 입력, 체크박스 조작 뒤의 요청 건수와 대화 쌍 수를 적습니다.

5월 21일 금요일 오후 4시에 가온관 201호에서 소리채집 동아리 첫 모임을 엽니다. 신입 회원은 이어폰을 챙겨 오세요. 진행자는 모임 20분 전에 도착해 녹음기를 책상마다 한 대씩 놓아 주세요.

<details><summary>모범 예시와 확인 기준</summary><pre><code>첫 입력  → 요청 1건 · 저장된 대화 1쌍
후속 질문 → 요청 2건 · 저장된 대화 2쌍
체크박스 다시 켜기 → 요청 2건 · 저장된 대화 2쌍</code></pre><p>공지의 오후 4시에서 20분을 빼면 오후 3시 40분입니다. 체크박스만 만졌는데 건수가 늘었다면 send가 if question 안에 있는지 확인하세요. 답은 한 번인데 메시지가 두 번 보인다면 기존 기록을 그리는 위치와 새 답을 그리는 위치를 살펴보세요.</p></details>

## 13일 · 동료의 공지로 후속 요청과 초기화 시험하기

공지를 두 문장으로 정리한 뒤 “한 문장으로 줄여 줘”라고 보내 보세요. 학생이 결정할 것은 어느 사실을 짧은 답에도 남길지입니다. 제출 마감이 중요한 공지라면 막연하게 “짧게”라고만 쓰는 대신 “마감과 해야 할 일을 남겨 한 문장으로 써 줘”라고 요청할 수 있습니다. 짝은 처음 보는 공지의 사실이 유지됐는지 확인합니다.

### 실습 · 새 대화 버튼 완성하기

12일의 화면 파일을 만드는 셀에서 ‘13일 과제’ 주석 자리를 고치세요. 버튼을 눌렀을 때 메시지 목록과 요청 수를 둘 다 초기화하세요. 기존 기록을 그리기 전에 처리해야 그 실행부터 빈 화면이 보입니다. 화면 파일 셀을 다시 실행해 저장하고 브라우저에서 변경된 앱을 다시 실행하세요. Streamlit이 변경을 감지해 Rerun을 물으면 그 버튼을 누르세요.

<details><summary>모범 예시와 확인 기준</summary><pre><code>if st.button(&quot;새 대화&quot;):
    st.session_state.messages = []  # 다음 질문에 보낼 이력도 비웁니다.
    st.session_state.calls = 0</code></pre><p>체크박스를 켰을 때 요청 0건·대화 0쌍이 보여야 합니다. 화면에서 글만 지우고 messages를 비우지 않으면 다음 호출에 지난 내용이 들어갑니다.</p></details>

### 동료가 처음 보는 C5로 시연하기

1. 빈 대화에서 C5 원문과 “신입 회원용으로 두 문장으로 정리해 줘”를 한 메시지로 보냅니다.
2. “해야 할 일과 마감을 남겨 한 문장으로 줄여 줘”만 보냅니다. 원문과 나란히 두고 5월 28일 금요일 오전 11시, 동아리 게시판 제출, 파일 제목에 녹음 장소 표시가 남았는지 확인하세요.
3. 새 대화를 누른 뒤 2번 문장만 보내세요. 원문을 다시 달라는 답을 받았는지 기록합니다. 모델이 구체적인 마감을 제시했다면 실제로 보낸 정보와 대조하세요.

C5 원문: 5월 28일 금요일 오전 11시까지 지난 모임에서 녹음한 소리를 동아리 게시판에 올려 주세요. 파일 제목에 녹음 장소를 넣어 주세요. 다음 주 모임에서 함께 들을 파일은 진행자가 두 개 고릅니다.

<details><summary>모범 예시와 확인 기준</summary><pre><code>학생이 원문을 읽고 만든 한 문장 예시
5월 28일 금요일 오전 11시까지 파일 제목에 녹음 장소를 적어 지난 모임의 녹음 파일을 동아리 게시판에 올려 주세요.</code></pre><p>첫 두 요청은 2건·2쌍입니다. 버튼 직후는 0건·0쌍이고, 마지막 질문 후에는 1건·1쌍입니다. 날짜를 모르는 새 대화에서 원문을 요청하는 응답도 이 실습의 성공 사례입니다.</p></details>

수업을 마쳤을 때 아래 두 줄을 노트북의 새 코드 셀에 복사해 실행하세요. 이 노트북으로 띄운 서버가 종료됩니다. 다음 문서 챗봇 수업에는 notice_chat.py와 notice_app.py를 같은 폴더에 보관해 가져갑니다.

```python
server.terminate()
server.wait()
```